# Carga Optimizada del Dataset Favorita Train - Manejo de Memoria

## Objetivo
Cargar y limpiar eficientemente el dataset `train.csv` (~3.6M filas) para análisis de forecasting de demanda sin saturar la memoria disponible.

### Problemas a Resolver:
- Valores booleanos como strings ("False", "True") en columna `onpromotion`
- Dataset muy grande para cargar completo en memoria
- Necesidad de optimización de tipos de datos para reducir footprint de memoria

## Sección 1: Importar Librerías y Configurar Rutas

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

# Configurar opciones de pandas para mejor visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# ============================================================================
# CONSTRUIR RUTAS RELATIVAS ROBUSTAS CON PATHLIB
# ============================================================================

# Obtener ruta del notebook actual
notebook_path = Path.cwd()

# Navegar hacia raíz del proyecto (desde notebooks/01_eda hacia segundo proyecto)
# notebook_path: c:\...\segundo proyecto\notebooks\01_eda
# project_root: c:\...\segundo proyecto
project_root = Path(notebook_path).parents[2]

# Definir rutas a carpetas clave
data_raw_path = project_root / "data" / "raw"
data_processed_path = project_root / "data" / "processed"

# Archivos específicos
train_file = data_raw_path / "train.csv"

# Validar existencia
assert train_file.exists(), f"❌ Archivo no encontrado: {train_file}"

print(f"✓ Proyecto raíz: {project_root}")
print(f"✓ Archivo a cargar: {train_file}")
print(f"✓ Tamaño archivo: {train_file.stat().st_size / 1024**3:.2f} GB\n")

## Sección 2: Preview del CSV Raw - Inspeccionar Estructura

In [ ]:
# Leer solo las primeras 100 filas para inspeccionar estructura y tipos
df_sample = pd.read_csv(train_file, nrows=100)

print("=" * 80)
print("PREVIEW DEL DATASET - PRIMERAS 100 FILAS")
print("=" * 80)
print(f"\nColumnas: {df_sample.columns.tolist()}")
print(f"Shape muestra: {df_sample.shape}")
print(f"\nTipos de datos (inferidos):\n{df_sample.dtypes}")
print(f"\nPrimeras 5 filas:\n{df_sample.head()}")
print(f"\nÚltimas 5 filas:\n{df_sample.tail()}")

# Inspeccionar valores únicos en onpromotion (donde sabemos hay problemas)
print(f"\n⚠️  Valores únicos en 'onpromotion':")
print(df_sample['onpromotion'].unique())
print(f"Tipo en muestra: {df_sample['onpromotion'].dtype}")

## Sección 3: Definir Dtype Mapping y Conversores Booleanos

In [ ]:
# ============================================================================
# DEFINIR TIPOS DE DATOS OPTIMIZADOS
# ============================================================================

# Especificar tipos de datos para reducir huella de memoria
# Usar tipos más pequeños donde sea posible
dtype_mapping = {
    'store_nbr': 'uint8',        # 0-54: cabe en uint8 (0-255)
    'product_id': 'uint32',      # ID de producto (hasta 4 billones)
    'sales': 'float32',          # Ventas: float32 es suficiente
    # 'date' se convertirá a datetime después
    # 'onpromotion' requiere conversor especial
}

# ============================================================================
# DEFINIR CONVERSOR PARA VALORES BOOLEANOS COMO STRINGS
# ============================================================================

def convert_onpromotion(value):
    """
    Convierte valores en columna onpromotion a entero (0 o 1)
    Maneja: 'False'/'True' (strings), 0/1 (números), NaN (None)
    """
    if pd.isna(value) or value == '':
        return 0  # Tratar NaN como sin promoción
    if isinstance(value, str):
        return 1 if value.strip().lower() == 'true' else 0
    return int(value)

# Definir función de conversión para fechas
def parse_date(date_str):
    """Convertir string a datetime - ejecutado en chunks"""
    return pd.to_datetime(date_str, format='%Y-%m-%d', errors='coerce')

print("✓ Dtype mapping y conversores definidos")
print(f"  Tipos a aplicar: {list(dtype_mapping.keys())}")

## Sección 4: Cargar Dataset con Chunking y Optimización de Memoria

In [ ]:
# ============================================================================
# CARGAR DATASET EN CHUNKS PARA EVITAR SATURACIÓN DE MEMORIA
# ============================================================================

print("=" * 80)
print("INICIANDO CARGA EN CHUNKS")
print("=" * 80)

# Configurar parámetros de chunking
chunk_size = 500_000  # Leer 500k filas por chunk
chunks_list = []
chunk_count = 0

# Leer archivo en chunks
print(f"\nLeyendo {train_file.name} en chunks de {chunk_size:,} filas...")
print("Aplicando conversiones: date → datetime, onpromotion → int\n")

try:
    for chunk in pd.read_csv(
        train_file,
        chunksize=chunk_size,
        dtype=dtype_mapping,
        converters={'onpromotion': convert_onpromotion},
        # parse_dates=['date'],  # Mejor hacerlo después de chunks concatenados
    ):
        chunk_count += 1
        
        # Convertir date a datetime en cada chunk
        chunk['date'] = pd.to_datetime(chunk['date'], format='%Y-%m-%d', errors='coerce')
        
        chunks_list.append(chunk)
        
        # Mostrar progreso
        rows_processed = chunk_count * chunk_size
        print(f"  ✓ Chunk {chunk_count}: {len(chunk):,} filas | Total procesadas: {rows_processed:,}")
        
        # Liberar memoria si es necesario
        if chunk_count % 3 == 0:
            print(f"    [Memory checkpoint after {chunk_count} chunks]")
    
    # Concatenar todos los chunks en un solo DataFrame
    print(f"\nConcatenando {len(chunks_list)} chunks...")
    df_train = pd.concat(chunks_list, ignore_index=True)
    print(f"✓ Dataset concatenado: {df_train.shape}")
    
except Exception as e:
    print(f"❌ Error durante carga en chunks: {e}")
    raise

print("\n✓ CARGA COMPLETADA EXITOSAMENTE")

## Sección 5: Validar Tipos de Datos y Calidad del Dataset

In [ ]:
# ============================================================================
# VALIDAR TIPOS DE DATOS Y ESTRUCTURA DEL DATASET
# ============================================================================

print("=" * 80)
print("VALIDACIÓN Y ANÁLISIS DEL DATASET CARGADO")
print("=" * 80)

# 1. Shape y tipos
print(f"\n1. DIMENSIONES Y TIPOS DE DATOS:")
print(f"   Shape: {df_train.shape} ({df_train.shape[0]:,} filas, {df_train.shape[1]} columnas)")
print(f"\n   Tipos de datos:")
print(df_train.dtypes)

# 2. Memoria
print(f"\n2. CONSUMO DE MEMORIA:")
memoria_mb = df_train.memory_usage(deep=False).sum() / 1024**2
print(f"   Total: {memoria_mb:.2f} MB")
for col in df_train.columns:
    mem = df_train[col].memory_usage(deep=False) / 1024**2
    print(f"   {col:<15} {mem:>8.4f} MB")

# 3. Análisis de valores nulos
print(f"\n3. ANÁLISIS DE VALORES NULOS:")
nulos = df_train.isnull().sum()
porcentaje_nulos = (nulos / len(df_train)) * 100
tabla_nulos = pd.DataFrame({
    'Columna': df_train.columns,
    'Valores Nulos': nulos.values,
    'Porcentaje (%)': porcentaje_nulos.values
}).sort_values('Porcentaje (%)', ascending=False)
print(tabla_nulos.to_string(index=False))

# 4. Validar conversión de onpromotion
print(f"\n4. VALIDACIÓN - COLUMNA 'onpromotion':")
print(f"   Valores únicos: {df_train['onpromotion'].unique()}")
print(f"   Tipo: {df_train['onpromotion'].dtype}")
print(f"   Distribución:")
print(df_train['onpromotion'].value_counts().sort_index())

# 5. Rango temporal
print(f"\n5. RANGO TEMPORAL:")
print(f"   Fecha mínima: {df_train['date'].min()}")
print(f"   Fecha máxima: {df_train['date'].max()}")
print(f"   Días únicos: {df_train['date'].nunique()}")

# 6. Estadísticas básicas
print(f"\n6. ESTADÍSTICAS DESCRIPTIVAS:")
print(df_train.describe())

## Sección 6: Exportar Dataset Limpio y Optimizado

In [ ]:
# ============================================================================
# EXPORTAR DATASET LIMPIO A FORMATOS OPTIMIZADOS
# ============================================================================

print("=" * 80)
print("EXPORTANDO DATASET LIMPIO")
print("=" * 80)

# Crear nombre de archivo con timestamp
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# ---- OPCIÓN 1: EXPORTAR A PARQUET (Recomendado: muy comprimido, tipado) ----
parquet_file = data_processed_path / f"train_cleaned.parquet"
print(f"\n1. Exportando a Parquet (muy comprimido)...")
df_train.to_parquet(
    parquet_file,
    index=False,
    compression='snappy',  # Buena relación compresión/velocidad
    engine='pyarrow'
)
parquet_size = parquet_file.stat().st_size / 1024**2
print(f"   ✓ Archivo guardado: {parquet_file}")
print(f"   ✓ Tamaño: {parquet_size:.2f} MB")
print(f"   ✓ Ratio compresión: {(parquet_size / memoria_mb) * 100:.1f}%")

# ---- OPCIÓN 2: EXPORTAR A CSV COMPRIMIDO (Compatible, más lento) ----
csv_gz_file = data_processed_path / f"train_cleaned.csv.gz"
print(f"\n2. Exportando a CSV comprimido (.gz)...")
df_train.to_csv(
    csv_gz_file,
    index=False,
    compression='gzip'
)
csv_size = csv_gz_file.stat().st_size / 1024**2
print(f"   ✓ Archivo guardado: {csv_gz_file}")
print(f"   ✓ Tamaño: {csv_size:.2f} MB")

# ---- OPCIÓN 3: GUARDAR METADATA ----
metadata = {
    'rows': len(df_train),
    'columns': list(df_train.columns),
    'dtypes': {col: str(dtype) for col, dtype in df_train.dtypes.items()},
    'date_range': {
        'min': str(df_train['date'].min()),
        'max': str(df_train['date'].max())
    },
    'memory_mb': memoria_mb,
    'parquet_file': str(parquet_file),
    'csv_gz_file': str(csv_gz_file),
    'exported_at': timestamp
}

import json
metadata_file = data_processed_path / "train_metadata.json"
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f"\n3. Metadata exportado a: {metadata_file}")

print("\n" + "=" * 80)
print("✓ EXPORTACIÓN COMPLETADA")
print("=" * 80)
print(f"\nPróximos pasos:")
print(f"  - Usar train_cleaned.parquet para análisis rápido")
print(f"  - El archivo es tipo-seguro y mucho más pequeño")
print(f"  - Para compartir: usar CSV.gz si necesitas compatibilidad máxima")